<a href="https://colab.research.google.com/github/WingmanShi/library_data_generator/blob/main/Copy_of_library_data_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# Community library data: create your own dataset (individual assignment)

You will work with data from a fictional community library service, covering the calendar year 2025. **Each student gets their own version.** The data are generated from your student ID, so:

- the same ID always gives the same data, so you can come back and re-create it;
- different IDs give different rows, values and problems;
- the layout of one table can differ from a classmate's, so check your data dictionary rather than relying on someone else's column names.

The files are exactly as exported from the library's systems. Checking them before you rely on them is part of the task.

Run the cells below in order.

## Step 1: Add the generator

This cell saves the data generator to the notebook's files. You do not need to edit it. Run it once.

In [11]:
%%writefile generate_library_data_by_id.py
#!/usr/bin/env python3
"""
Community-library dataset, personalised by student ID (fictional data).

Student use (Colab or terminal):
    from generate_library_data_by_id import generate
    generate(input("Enter your student ID: "))
  or:  python generate_library_data_by_id.py          (prompts for the ID)
       python generate_library_data_by_id.py 1234567

What the ID does
  1. SEED: the ID is hashed into a random seed, so the same ID always gives the
     same dataset, and different IDs give different rows, values and mess.
  2. LAYOUT VARIANT: the last digit of the ID selects one of five layouts.
     The underlying data and the injected problems are the same in kind and
     amount for everyone; the variant changes how one table is laid out.
        last digit 1, 2 -> A  baseline layout
        last digit 3, 4 -> B  fines stored as whole pence (fine_pence)
        last digit 5, 6 -> C  loans split over two files (first/second half of year)
        last digit 7, 8 -> D  loans give branch NAME (branch), not branch_id
        last digit 9, 0 -> E  item type and category merged (classification)

Marker use
    python generate_library_data_by_id.py 1234567 --marker
    python generate_library_data_by_id.py --ids ids.txt --marker   (one ID per line)
  writes marker_lists/marker_<ID>.md (issue counts, reference numbers) and, for
  --ids, marker_lists/variant_summary.csv.
  NOTE: students can read this script. It does not write the marker list unless
  asked, but the issue types are visible in the code. See the staff notes.
"""
import argparse
import hashlib
import os
import re
import sys
import tempfile
from datetime import date, timedelta

import numpy as np
import pandas as pd

VERSION = "library-v1"
START, END = date(2025, 1, 1), date(2025, 12, 31)
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"]
DATE_FORMAT_PROBS = [0.78, 0.17, 0.05]
VARIANT_BY_DIGIT = {"1": "A", "2": "A", "3": "B", "4": "B", "5": "C", "6": "C",
                    "7": "D", "8": "D", "9": "E", "0": "E"}
VARIANT_TEXT = {
    "A": "Baseline layout.",
    "B": "Fines stored as whole pence in loans.fine_pence (integers), not pounds.",
    "C": "Loans split over loans_2025_H1.csv (Jan-Jun) and loans_2025_H2.csv (Jul-Dec); "
         "H2 has its columns in a different order.",
    "D": "loans has the branch NAME in column 'branch' instead of branch_id; join to branches on branch_name.",
    "E": "items has one 'classification' column (\"Type / Category\") instead of item_type and category.",
}


# ------------------------------------------------------------------ helpers
def seed_from_id(student_id):
    digest = hashlib.sha256(f"{VERSION}:{student_id.strip().upper()}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


def variant_from_id(student_id):
    digits = re.findall(r"\d", student_id)
    if not digits:
        raise ValueError("Student ID must contain at least one digit.")
    return VARIANT_BY_DIGIT[digits[-1]]


def clean_id(student_id):
    sid = str(student_id).strip()
    if not sid:
        raise ValueError("Student ID is empty.")
    return sid


def fmt_date(d, f):
    return "" if d is None or pd.isna(d) else d.strftime(f)


def parse_date(s):
    s = str(s).strip()
    if not s or s == "nan":
        return pd.NaT
    if "/" in s:
        return pd.to_datetime(s, format="%d/%m/%Y")
    if any(ch.isalpha() for ch in s):
        return pd.to_datetime(s, format="%d %b %Y")
    return pd.to_datetime(s, format="%Y-%m-%d")


# ------------------------------------------------------- baseline generation
def build_baseline(seed, out):
    """Writes baseline-layout CSVs into `out`; returns the injected-issue log."""
    rng = np.random.default_rng(seed)
    os.makedirs(out, exist_ok=True)
    log = {}

    def pick_idx(n, k):
        return rng.choice(n, size=k, replace=False)

    # ---- branches
    branch_weights = np.array([0.22, 0.12, 0.15, 0.08, 0.14, 0.10, 0.11, 0.08])
    branches = pd.DataFrame({
        "branch_id": [f"B{i:02d}" for i in range(1, 9)],
        "branch_name": ["Abbeygate", "Brookfield", "Castlemere", "Dunmore",
                        "Elmstead", "Fairholme", "Greystone", "Harlow Green"],
        "area": ["North", "North", "East", "East", "South", "South", "West", "West"],
        "opened_year": [1962, 1974, 1988, 1995, 1971, 2003, 1958, 2011],
    })

    # ---- items
    N_ITEMS = 1200
    adjectives = ["Quiet", "Last", "Hidden", "Silver", "Northern", "Broken", "Golden",
                  "Winter", "Distant", "Little", "Secret", "Burning", "Paper", "Wild"]
    nouns = ["River", "Garden", "Letters", "Harbour", "Orchard", "Kingdom", "Season",
             "Map", "Bridge", "Lighthouse", "Stranger", "Engine", "Forest", "Promise"]
    item_types = rng.choice(["Book", "DVD", "Audiobook"], N_ITEMS, p=[0.75, 0.15, 0.10])
    category_by_type = {
        "Book": (["Fiction", "Non-fiction", "Children", "Reference"], [0.40, 0.30, 0.25, 0.05]),
        "DVD": (["Film", "Children"], [0.65, 0.35]),
        "Audiobook": (["Fiction", "Non-fiction"], [0.60, 0.40]),
    }
    cost_range = {"Book": (6, 25), "DVD": (8, 20), "Audiobook": (12, 30)}
    categories, costs = [], []
    for t in item_types:
        cats, probs = category_by_type[t]
        categories.append(rng.choice(cats, p=probs))
        lo, hi = cost_range[t]
        costs.append(round(float(rng.uniform(lo, hi)), 2))
    items = pd.DataFrame({
        "item_id": [f"I{i:04d}" for i in range(1, N_ITEMS + 1)],
        "title": [f"The {rng.choice(adjectives)} {rng.choice(nouns)}" for _ in range(N_ITEMS)],
        "item_type": item_types,
        "category": categories,
        "replacement_cost": costs,
        "home_branch_id": rng.choice(branches["branch_id"], N_ITEMS, p=branch_weights),
    })
    clean_items = items.copy()

    # ---- members
    N_MEMBERS = 800
    age_bands = ["Under 12", "12-17", "18-34", "35-59", "60+"]
    bands = rng.choice(age_bands, N_MEMBERS, p=[0.14, 0.12, 0.22, 0.30, 0.22])

    def membership(b):
        if b in ("Under 12", "12-17"):
            return "Junior"
        if b == "60+" and rng.random() < 0.6:
            return "Concession"
        return "Standard"

    join_days = rng.integers(0, (date(2025, 10, 31) - date(2015, 1, 1)).days, N_MEMBERS)
    letters = np.array(list("ABDEFGHJLNPQRSTUWXYZ"))
    members = pd.DataFrame({
        "member_id": [f"M{i:05d}" for i in range(1, N_MEMBERS + 1)],
        "join_date": [date(2015, 1, 1) + timedelta(int(d)) for d in join_days],
        "age_band": bands,
        "membership_type": [membership(b) for b in bands],
        "home_branch_id": rng.choice(branches["branch_id"], N_MEMBERS, p=branch_weights),
        "postcode": [f"ZZ{rng.integers(1, 10)} {rng.integers(1, 10)}{''.join(rng.choice(letters, 2))}"
                     for _ in range(N_MEMBERS)],
    })
    clean_members = members.copy()
    activity = rng.gamma(1.2, 1.0, N_MEMBERS)
    activity = activity / activity.sum()

    # ---- loans
    N_LOANS = 6000
    cat_prefs = {
        "Under 12": {"Children": .70, "Film": .10, "Fiction": .10, "Non-fiction": .05, "Reference": .05},
        "12-17": {"Fiction": .40, "Children": .20, "Film": .20, "Non-fiction": .15, "Reference": .05},
        "18-34": {"Fiction": .35, "Film": .25, "Non-fiction": .30, "Children": .02, "Reference": .08},
        "35-59": {"Fiction": .30, "Non-fiction": .35, "Film": .15, "Children": .12, "Reference": .08},
        "60+": {"Fiction": .40, "Non-fiction": .35, "Film": .12, "Children": .05, "Reference": .08},
    }
    items_by_cat = {}
    for c, grp in clean_items.groupby("category"):
        w = rng.pareto(2.0, len(grp)) + 1
        items_by_cat[c] = (grp.reset_index(drop=True), w / w.sum())

    days = [START + timedelta(d) for d in range((END - START).days + 1)]
    day_w = np.array([(2.0 if d.weekday() == 5 else 0.6 if d.weekday() == 6 else 1.0)
                      * (1.4 if d.month in (7, 8) else 0.8 if d.month == 12 else 1.0)
                      for d in days])
    loan_dates = sorted(rng.choice(len(days), N_LOANS, p=day_w / day_w.sum()))
    period = {"Book": 21, "DVD": 7, "Audiobook": 14}
    fine_rate = {"Book": 0.20, "Audiobook": 0.20, "DVD": 1.00}

    rows = []
    for n, di in enumerate(loan_dates, start=1):
        ld = days[di]
        m = clean_members.iloc[rng.choice(N_MEMBERS, p=activity)]
        prefs = cat_prefs[m["age_band"]]
        cat = rng.choice(list(prefs), p=list(prefs.values()))
        grp, w = items_by_cat[cat]
        it = grp.iloc[rng.choice(len(grp), p=w)]
        branch = m["home_branch_id"] if rng.random() < 0.75 else rng.choice(branches["branch_id"], p=branch_weights)
        p = period[it["item_type"]]
        due = ld + timedelta(p)
        still_out = ld > END - timedelta(21) and rng.random() < 0.6
        lost = (not still_out) and rng.random() < 0.015
        ret, fine = None, np.nan
        if not still_out and not lost:
            if rng.random() < 0.84:
                days_out = int(rng.integers(1, p + 1))
            else:
                days_out = p + min(int(rng.geometric(0.15)), 60)
            ret = ld + timedelta(days_out)
            late = max(0, days_out - p)
            fine = 0.0 if it["category"] == "Children" else min(10.0, round(late * fine_rate[it["item_type"]], 2))
        rows.append([f"L{n:06d}", m["member_id"], it["item_id"], branch, ld, due, ret, fine])
    loans = pd.DataFrame(rows, columns=["loan_id", "member_id", "item_id", "branch_id",
                                        "loan_date", "due_date", "return_date", "fine_charged"])

    # ================================================================ inject mess
    # ---- items
    n = len(items)
    variants = {"Fiction": ["fiction", "FICTION", "Fiction "],
                "Non-fiction": ["Non fiction", "nonfiction", "Non-Fiction", "non-fiction "],
                "Children": ["Childrens", "Children's", "children"],
                "Reference": ["reference", "Ref."], "Film": ["film", "Films"]}
    idx = pick_idx(n, int(0.12 * n))
    items["category"] = items["category"].astype(object)
    for i in idx:
        items.at[i, "category"] = rng.choice(variants[items.at[i, "category"]])
    log["items_category_variants"] = len(idx)
    items["replacement_cost"] = items["replacement_cost"].astype(object)
    idx = pick_idx(n, int(0.06 * n))
    for i in idx:
        items.at[i, "replacement_cost"] = f"£{items.at[i, 'replacement_cost']:.2f}"
    log["items_cost_as_text_with_pound_sign"] = len(idx)
    rest = np.setdiff1d(np.arange(n), idx)
    idx2 = rng.choice(rest, int(0.02 * n), replace=False)
    items.loc[idx2, "replacement_cost"] = np.nan
    log["items_cost_missing"] = len(idx2)

    # ---- members
    n = len(members)
    band_variants = {"Under 12": ["under 12", "<12"], "12-17": ["12 - 17", "12 to 17"],
                     "18-34": ["18 - 34", "18 to 34"], "35-59": ["35 - 59", "35 to 59"],
                     "60+": ["60 plus", "60+ "]}
    idx = pick_idx(n, int(0.08 * n))
    for i in idx:
        members.at[i, "age_band"] = rng.choice(band_variants[members.at[i, "age_band"]])
    log["members_age_band_variants"] = len(idx)

    jd_fmt = rng.choice(len(DATE_FORMATS), n, p=DATE_FORMAT_PROBS)
    members["join_date"] = [fmt_date(d, DATE_FORMATS[f]) for d, f in zip(members["join_date"], jd_fmt)]
    log["members_join_date_not_iso"] = int((jd_fmt != 0).sum())

    pc = members["postcode"].tolist()
    low = pick_idx(n, int(0.15 * n))
    for i in low:
        pc[i] = pc[i].lower()
    nospace = pick_idx(n, int(0.10 * n))
    for i in nospace:
        pc[i] = pc[i].replace(" ", "")
    miss = pick_idx(n, int(0.04 * n))
    for i in miss:
        pc[i] = np.nan
    members["postcode"] = pc
    log["members_postcode_lowercase"] = len(low)
    log["members_postcode_no_space"] = len(nospace)
    log["members_postcode_missing"] = len(miss)

    exact = members.iloc[pick_idx(n, int(0.03 * n))]
    cand = members[(jd_fmt == 0) & members["postcode"].notna()
                   & (members["postcode"].astype(str).str.upper() == members["postcode"].astype(str))]
    near = cand.sample(8, random_state=int(rng.integers(0, 2**31 - 1))).copy()
    near["join_date"] = [pd.to_datetime(d).strftime("%d/%m/%Y") for d in near["join_date"]]
    near["postcode"] = near["postcode"].astype(str).str.lower()
    members = pd.concat([members, exact, near]).sort_values("member_id", kind="mergesort").reset_index(drop=True)
    log["members_exact_duplicate_rows"] = len(exact)
    log["members_near_duplicate_rows"] = len(near)

    # ---- loans
    n = len(loans)
    returned = loans.index[loans["return_date"].notna()].to_numpy()
    bad_ret = rng.choice(returned, 12, replace=False)
    for i in bad_ret:
        loans.at[i, "return_date"] = loans.at[i, "loan_date"] - timedelta(int(rng.integers(1, 11)))
    log["loans_return_before_loan"] = len(bad_ret)

    pos_fine = loans.index[(loans["fine_charged"] > 0) & ~loans.index.isin(bad_ret)].to_numpy()
    pick = rng.choice(pos_fine, 9, replace=False)
    big, neg = pick[:3], pick[3:]
    loans.loc[big, "fine_charged"] = loans.loc[big, "fine_charged"] * 100
    loans.loc[neg, "fine_charged"] = -loans.loc[neg, "fine_charged"]
    log["loans_fine_100x_outliers"] = len(big)
    log["loans_fine_negative"] = len(neg)

    d_fmt = rng.choice(len(DATE_FORMATS), n, p=DATE_FORMAT_PROBS)
    for col in ["loan_date", "due_date", "return_date"]:
        loans[col] = [fmt_date(d, DATE_FORMATS[f]) for d, f in zip(loans[col], d_fmt)]
    log["loans_rows_with_non_iso_dates"] = int((d_fmt != 0).sum())

    fine_txt = []
    n_pound = n_blank_zero = 0
    for v in loans["fine_charged"]:
        if pd.isna(v):
            fine_txt.append("")
        elif v == 0:
            if rng.random() < 0.10:
                fine_txt.append(""); n_blank_zero += 1
            else:
                fine_txt.append("0.00")
        elif v > 0 and rng.random() < 0.07:
            fine_txt.append(f"£{v:.2f}"); n_pound += 1
        else:
            fine_txt.append(f"{v:.2f}")
    loans["fine_charged"] = fine_txt
    log["loans_fine_with_pound_sign"] = n_pound
    log["loans_zero_fine_left_blank"] = n_blank_zero

    bid = loans["branch_id"].tolist()
    low = pick_idx(n, int(0.02 * n))
    for i in low:
        bid[i] = bid[i].lower()
    rest = np.setdiff1d(np.arange(n), low)
    spc = rng.choice(rest, int(0.01 * n), replace=False)
    for i in spc:
        bid[i] = " " + bid[i]
    loans["branch_id"] = bid
    log["loans_branch_id_lowercase"] = len(low)
    log["loans_branch_id_leading_space"] = len(spc)

    o_mem = pick_idx(n, int(0.01 * n))
    loans.loc[o_mem, "member_id"] = [f"M{rng.integers(901, 999):05d}" for _ in o_mem]
    rest = np.setdiff1d(np.arange(n), o_mem)
    o_item = rng.choice(rest, int(0.005 * n), replace=False)
    loans.loc[o_item, "item_id"] = [f"I{rng.integers(1201, 1300):04d}" for _ in o_item]
    log["loans_member_id_not_in_members"] = len(o_mem)
    log["loans_item_id_not_in_items"] = len(o_item)

    dups = loans.iloc[pick_idx(n, int(0.01 * n))]
    loans = pd.concat([loans, dups]).sort_values("loan_id", kind="mergesort").reset_index(drop=True)
    log["loans_exact_duplicate_rows"] = len(dups)

    branches.to_csv(f"{out}/branches.csv", index=False)
    members.to_csv(f"{out}/members.csv", index=False)
    items.to_csv(f"{out}/items.csv", index=False)
    loans.to_csv(f"{out}/loans.csv", index=False)
    return log


# ---------------------------------------------------- reference numbers (marker)
def reference_numbers(base):
    raw = pd.read_csv(f"{base}/loans.csv", dtype=str, keep_default_na=False)
    c = raw.drop_duplicates().copy()
    c["branch_id"] = c["branch_id"].str.strip().str.upper()
    for col in ["loan_date", "due_date", "return_date"]:
        c[col] = c[col].map(parse_date)
    mem_ids = set(pd.read_csv(f"{base}/members.csv", dtype=str)["member_id"])
    item_ids = set(pd.read_csv(f"{base}/items.csv", dtype=str)["item_id"])
    ref = {
        "raw_loan_rows": len(raw),
        "loan_rows_after_dropping_exact_duplicates": len(c),
        "loans_with_member_not_in_members": int((~c["member_id"].isin(mem_ids)).sum()),
        "loans_with_item_not_in_items": int((~c["item_id"].isin(item_ids)).sum()),
    }
    ok = c[c["return_date"].notna() & (c["return_date"] >= c["loan_date"])]
    ref["late_return_share_of_valid_returns"] = round(float((ok["return_date"] > ok["due_date"]).mean()), 3)
    by_branch = c["branch_id"].value_counts().sort_index()
    by_month = c["loan_date"].dt.to_period("M").value_counts().sort_index()
    return ref, by_branch, by_month


# ------------------------------------------------------------ layout variants
def apply_variant(variant, base, out):
    """Reads baseline CSVs from `base`, writes the student's layout to `out`.
    Returns (files written, log keys that no longer apply, schema notes)."""
    os.makedirs(out, exist_ok=True)
    rd = lambda f: pd.read_csv(f"{base}/{f}.csv", dtype=str, keep_default_na=False)
    branches, members, items, loans = rd("branches"), rd("members"), rd("items"), rd("loans")
    dropped = []

    if variant == "B":
        def to_pence(s):
            s = s.strip().replace("£", "")
            return "" if s == "" else str(int(round(float(s) * 100)))
        loans["fine_charged"] = loans["fine_charged"].map(to_pence)
        loans = loans.rename(columns={"fine_charged": "fine_pence"})
        dropped = ["loans_fine_with_pound_sign"]
    elif variant == "D":
        name = dict(zip(branches["branch_id"], branches["branch_name"]))

        def to_name(v):
            core = name[v.strip().upper()]
            if v != v.strip():
                return " " + core
            return core.lower() if v == v.lower() else core
        loans["branch_id"] = loans["branch_id"].map(to_name)
        loans = loans.rename(columns={"branch_id": "branch"})
    elif variant == "E":
        items.insert(2, "classification", items["item_type"] + " / " + items["category"])
        items = items.drop(columns=["item_type", "category"])

    files = {"branches": branches, "members": members, "items": items}
    if variant == "C":
        half = loans["loan_date"].map(parse_date).dt.month <= 6
        h1, h2 = loans[half], loans[~half]
        cols = list(h2.columns)
        cols[4], cols[6] = cols[6], cols[4]  # swap loan_date / return_date positions
        files["loans_2025_H1"] = h1
        files["loans_2025_H2"] = h2[cols]
    else:
        files["loans"] = loans
    for k, df in files.items():
        df.to_csv(f"{out}/{k}.csv", index=False)
    return list(files), dropped


def data_dictionary(variant, files):
    d = {
        "branches": "branch_id, branch_name, area, opened_year",
        "members": "member_id, join_date, age_band, membership_type (Junior / Standard / Concession), home_branch_id, postcode",
        "items": "item_id, title, item_type, category, replacement_cost (GBP), home_branch_id",
        "loans": "loan_id, member_id, item_id, branch_id (branch where the loan was made), loan_date, due_date, "
                 "return_date (blank = not returned), fine_charged (GBP)",
    }
    if variant == "E":
        d["items"] = "item_id, title, classification (item type / category), replacement_cost (GBP), home_branch_id"
    if variant == "B":
        d["loans"] = d["loans"].replace("fine_charged (GBP)", "fine_pence (whole pence)")
    if variant == "D":
        d["loans"] = d["loans"].replace("branch_id (branch where", "branch (name of the branch where")
    lines = ["# Data dictionary", "", "Fictional data for a community library service, calendar year 2025.", ""]
    for f in files:
        key = "loans" if f.startswith("loans") else f
        lines.append(f"- **{f}.csv**: {d[key]}")
    if variant == "C":
        lines.append("")
        lines.append("loans are split by loan date: H1 = January to June, H2 = July to December. "
                     "Check the column order in each file.")
    lines += ["", "The files are exactly as exported from the library's systems. "
                  "Checking them before you rely on them is part of the task.", ""]
    return "\n".join(lines)


# ---------------------------------------------------------------- marker list
def write_marker(path, sid, seed, variant, log, dropped, ref, by_branch, by_month):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# Marker's list: library data for student ID {sid} (PRIVATE)\n\n")
        f.write(f"Seed (from ID): {seed}. Layout variant: **{variant}**: {VARIANT_TEXT[variant]}\n\n")
        f.write("The issues are ordinary, not traps. Use this to judge how thorough a student's checking was, "
                "not as a checklist every student must complete.\n\n")
        f.write("## Injected issues\n\n")
        f.write("Counts are for the original rows. Duplicate rows were appended afterwards, so a few affected rows appear twice.\n\n")
        f.write("| Issue | Rows affected |\n| --- | --- |\n")
        for k, v in log.items():
            if k in dropped:
                f.write(f"| {k.replace('_', ' ')} | not applicable in variant {variant} (converted to pence) |\n")
            else:
                f.write(f"| {k.replace('_', ' ')} | {v} |\n")
        f.write("\n## Issues arising from the design of the data (no injection)\n\n")
        f.write("- Return date blank: either a loan still out (recent loans) or an item never returned (old loans). The data does not say which.\n")
        f.write("- Zero fines on all Children items (a fine-free category), plus blank fines on some loans.\n")
        f.write("- Slash dates (dd/mm/yyyy) are ambiguous if read as mm/dd/yyyy; the format is consistent within a row.\n")
        f.write("- Loan periods differ by item type (Book 21, Audiobook 14, DVD 7 days), so lateness depends on joining to items.\n\n")
        f.write("## Variant-specific step\n\n")
        f.write(f"- {VARIANT_TEXT[variant]}\n\n")
        f.write("## Reference numbers after one defensible cleaning approach\n\n")
        f.write("Approach: drop exact duplicate loan rows, trim and upper-case branch_id, parse dates by format. "
                "Figures are the same whichever layout the student received. "
                "Other defensible choices (for example how to treat orphan keys) will change some figures.\n\n")
        for k, v in ref.items():
            f.write(f"- {k.replace('_', ' ')}: {v}\n")
        f.write("\n### Loans per branch (after cleaning)\n\n| branch_id | loans |\n| --- | --- |\n")
        for b, v in by_branch.items():
            f.write(f"| {b} | {v} |\n")
        f.write("\n### Loans per month (after cleaning)\n\n| month | loans |\n| --- | --- |\n")
        for m_, v in by_month.items():
            f.write(f"| {m_} | {v} |\n")


# ------------------------------------------------------------------- public
def generate(student_id, out_dir=None, marker=False, marker_dir="marker_lists", variant=None, quiet=False):
    """Create this student's dataset. `variant` overrides the ID-derived layout (testing only)."""
    sid = clean_id(student_id)
    seed = seed_from_id(sid)
    variant = variant or variant_from_id(sid)
    out_dir = out_dir or f"library_data_{re.sub(r'[^A-Za-z0-9]', '', sid)}"
    with tempfile.TemporaryDirectory() as tmp:
        log = build_baseline(seed, tmp)
        files, dropped = apply_variant(variant, tmp, out_dir)
        with open(f"{out_dir}/data_dictionary.md", "w", encoding="utf-8") as f:
            f.write(data_dictionary(variant, files))
        if marker:
            ref, by_branch, by_month = reference_numbers(tmp)
            os.makedirs(marker_dir, exist_ok=True)
            write_marker(f"{marker_dir}/marker_{re.sub(r'[^A-Za-z0-9]', '', sid)}.md",
                         sid, seed, variant, log, dropped, ref, by_branch, by_month)
    if not quiet:
        print(f"Created {out_dir}/ with: " + ", ".join(f"{f}.csv" for f in files) + ", data_dictionary.md")
    return out_dir, variant


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("student_id", nargs="?")
    ap.add_argument("--ids", help="text file with one student ID per line (marker use)")
    ap.add_argument("--marker", action="store_true", help="also write the private marker list")
    ap.add_argument("--variant", choices=list("ABCDE"), help="override layout (testing only)")
    a = ap.parse_args()
    if a.ids:
        rows = []
        for line in open(a.ids, encoding="utf-8"):
            sid = line.strip()
            if sid:
                _, v = generate(sid, out_dir=f"all_students/library_data_{re.sub(r'[^A-Za-z0-9]', '', sid)}",
                                marker=a.marker, quiet=True)
                rows.append((sid, v))
        os.makedirs("marker_lists", exist_ok=True)
        pd.DataFrame(rows, columns=["student_id", "variant"]).to_csv("marker_lists/variant_summary.csv", index=False)
        print(f"Generated {len(rows)} datasets in all_students/")
    else:
        sid = a.student_id or input("Enter your student ID: ")
        generate(sid, marker=a.marker, variant=a.variant)


if __name__ == "__main__":
    main()

Overwriting generate_library_data_by_id.py


## Step 2: Create your data

Run the cell and type your student ID when asked. Use the ID exactly as it appears on your student card. A different ID gives different data, so check you typed it correctly.

In [12]:
from generate_library_data_by_id import generate

student_id = input("Enter your student ID: ")
data_dir, _ = generate(student_id)

Enter your student ID: 5785939
Created library_data_5785939/ with: branches.csv, members.csv, items.csv, loans.csv, data_dictionary.md


## Step 3: Read your data dictionary

This describes the files you have been given. Read it before you open the data.

In [13]:
print(open(f"{data_dir}/data_dictionary.md", encoding="utf-8").read())

# Data dictionary

Fictional data for a community library service, calendar year 2025.

- **branches.csv**: branch_id, branch_name, area, opened_year
- **members.csv**: member_id, join_date, age_band, membership_type (Junior / Standard / Concession), home_branch_id, postcode
- **items.csv**: item_id, title, classification (item type / category), replacement_cost (GBP), home_branch_id
- **loans.csv**: loan_id, member_id, item_id, branch_id (branch where the loan was made), loan_date, due_date, return_date (blank = not returned), fine_charged (GBP)

The files are exactly as exported from the library's systems. Checking them before you rely on them is part of the task.



## Step 4a: Load the files (optional - if you are working in Colab)

This loads every CSV file in your folder into a dictionary called `tables`, exactly as it is in the file, with no cleaning. Your file list may differ from a classmate's.

In [14]:
import glob
import os

import pandas as pd
from IPython.display import display

tables = {}
for path in sorted(glob.glob(f"{data_dir}/*.csv")):
    name = os.path.splitext(os.path.basename(path))[0]
    tables[name] = pd.read_csv(path)

for name, df in tables.items():
    print(f"\n=== {name}: {df.shape[0]:,} rows, {df.shape[1]} columns ===")
    display(df.head(3))


=== branches: 8 rows, 4 columns ===


,branch_id,branch_name,area,opened_year
0,B01,Abbeygate,North,1962
1,B02,Brookfield,North,1974
2,B03,Castlemere,East,1988



=== items: 1,200 rows, 5 columns ===


,item_id,title,classification,replacement_cost,home_branch_id
0,I0001,The Secret Map,Book / Children,£18.98,B03
1,I0002,The Quiet Season,Book / Fiction,21.18,B05
2,I0003,The Hidden River,Book / Non-fiction,£16.19,B01



=== loans: 6,060 rows, 8 columns ===


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged
0,L000001,M00106,I0813,B07,2025-01-01,2025-01-22,2025-01-07,0.00
1,L000002,M00374,I0132,B07,2025-01-01,2025-01-08,2025-01-07,0.00
2,L000002,M00374,I0132,B07,2025-01-01,2025-01-08,2025-01-07,0.00



=== members: 832 rows, 6 columns ===


,member_id,join_date,age_band,membership_type,home_branch_id,postcode
0,M00001,2021-12-12,Under 12,Junior,B06,ZZ7 7BQ
1,M00002,2021-11-19,60+,Concession,B03,ZZ4 7PZ
2,M00003,2018-11-14,Under 12,Junior,B08,ZZ3 4ZA


## 1. Check the original data

In [16]:

import os
import pandas as pd
import glob

# Find your generated dataset folder
data_dir = "library_data_5785939"

# Display original file information
print("MY ORIGINAL DATASET")
print("=" * 50)

for path in sorted(glob.glob(f"{data_dir}/*.csv")):

    filename = os.path.basename(path)

    df = pd.read_csv(path)

    print(f"\nFile: {filename}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Column names: {list(df.columns)}")

    print("\nFirst 3 rows:")
    display(df.head(3))


MY ORIGINAL DATASET

File: branches.csv
Rows: 8
Columns: 4
Column names: ['branch_id', 'branch_name', 'area', 'opened_year']

First 3 rows:


,branch_id,branch_name,area,opened_year
0,B01,Abbeygate,North,1962
1,B02,Brookfield,North,1974
2,B03,Castlemere,East,1988



File: items.csv
Rows: 1,200
Columns: 5
Column names: ['item_id', 'title', 'classification', 'replacement_cost', 'home_branch_id']

First 3 rows:


,item_id,title,classification,replacement_cost,home_branch_id
0,I0001,The Secret Map,Book / Children,£18.98,B03
1,I0002,The Quiet Season,Book / Fiction,21.18,B05
2,I0003,The Hidden River,Book / Non-fiction,£16.19,B01



File: loans.csv
Rows: 6,060
Columns: 8
Column names: ['loan_id', 'member_id', 'item_id', 'branch_id', 'loan_date', 'due_date', 'return_date', 'fine_charged']

First 3 rows:


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged
0,L000001,M00106,I0813,B07,2025-01-01,2025-01-22,2025-01-07,0.00
1,L000002,M00374,I0132,B07,2025-01-01,2025-01-08,2025-01-07,0.00
2,L000002,M00374,I0132,B07,2025-01-01,2025-01-08,2025-01-07,0.00



File: members.csv
Rows: 832
Columns: 6
Column names: ['member_id', 'join_date', 'age_band', 'membership_type', 'home_branch_id', 'postcode']

First 3 rows:


,member_id,join_date,age_band,membership_type,home_branch_id,postcode
0,M00001,2021-12-12,Under 12,Junior,B06,ZZ7 7BQ
1,M00002,2021-11-19,60+,Concession,B03,ZZ4 7PZ
2,M00003,2018-11-14,Under 12,Junior,B08,ZZ3 4ZA


## 2. Check data quality

I will check missing values, duplicate records, duplicate loan IDs, and the values in classification column.

In [17]:

import pandas as pd
import os

# Load the original CSV files
branches = pd.read_csv(os.path.join(data_dir, "branches.csv"))
items = pd.read_csv(os.path.join(data_dir, "items.csv"))
loans = pd.read_csv(os.path.join(data_dir, "loans.csv"))
members = pd.read_csv(os.path.join(data_dir, "members.csv"))

datasets = {
    "branches": branches,
    "items": items,
    "loans": loans,
    "members": members
}

# 1. Check missing values and duplicate rows
for name, df in datasets.items():
    print("\n" + "=" * 50)
    print("DATA QUALITY:", name.upper())
    print("=" * 50)

    print("Missing values:")
    print(df.isnull().sum())

    print("\nCompletely duplicated rows:", df.duplicated().sum())

# 2. Check duplicate loan IDs
print("\nDUPLICATE LOAN IDs")
print(loans[loans.duplicated(subset=["loan_id"], keep=False)]
      .sort_values("loan_id"))

# 3. Check item classifications
print("\nITEM CLASSIFICATIONS")
print(items["classification"].value_counts(dropna=False))

# 4. Check replacement cost formats
print("\nREPLACEMENT COST EXAMPLES")
print(items["replacement_cost"].head(20).to_list())

# 5. Check date ranges
print("\nLOAN DATE RANGE")
print("Earliest:", loans["loan_date"].min())
print("Latest:", loans["loan_date"].max())



DATA QUALITY: BRANCHES
Missing values:
branch_id      0
branch_name    0
area           0
opened_year    0
dtype: int64

Completely duplicated rows: 0

DATA QUALITY: ITEMS
Missing values:
item_id              0
title                0
classification       0
replacement_cost    24
home_branch_id       0
dtype: int64

Completely duplicated rows: 0

DATA QUALITY: LOANS
Missing values:
loan_id           0
member_id         0
item_id           0
branch_id         0
loan_date         0
due_date          0
return_date     247
fine_charged    773
dtype: int64

Completely duplicated rows: 60

DATA QUALITY: MEMBERS
Missing values:
member_id           0
join_date           0
age_band            0
membership_type     0
home_branch_id      0
postcode           34
dtype: int64

Completely duplicated rows: 24

DUPLICATE LOAN IDs
      loan_id member_id item_id branch_id   loan_date    due_date return_date  \
1     L000002    M00374   I0132       B07  2025-01-01  2025-01-08  2025-01-07   
2     L00000

Now, i will check the date formats for each file.

In [18]:

import pandas as pd
import re

# Identify the date columns in each dataset
date_columns = {
    "members": {
        "df": members,
        "columns": ["join_date"]
    },
    "loans": {
        "df": loans,
        "columns": ["loan_date", "due_date", "return_date"]
    }
}

# Identify common date formats without changing the original values
def identify_date_format(value):
    if pd.isna(value):
        return "Missing"

    value = str(value).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return "YYYY-MM-DD"

    elif re.fullmatch(r"\d{2}/\d{2}/\d{4}", value):
        return "DD/MM/YYYY or MM/DD/YYYY"

    elif re.fullmatch(r"\d{1,2} [A-Za-z]{3} \d{4}", value):
        return "DD Mon YYYY"

    else:
        return "Other / Unrecognised"

# Inspect each date column separately
for dataset_name, info in date_columns.items():

    print("\n" + "=" * 50)
    print("DATE FORMAT CHECK:", dataset_name.upper())
    print("=" * 50)

    df = info["df"]

    for column in info["columns"]:

        formats = df[column].apply(identify_date_format)

        print("\nColumn:", column)
        print(formats.value_counts(dropna=False))

        print("\nExamples:")
        print(df[column].dropna().unique()[:10])



DATE FORMAT CHECK: MEMBERS

Column: join_date
join_date
YYYY-MM-DD                  627
DD/MM/YYYY or MM/DD/YYYY    161
DD Mon YYYY                  44
Name: count, dtype: int64

Examples:
['2021-12-12' '2021-11-19' '2018-11-14' '09/07/2018' '03/01/2021'
 '2025-08-30' '30/05/2021' '2022-02-01' '2021-03-25' '2019-06-10']

DATE FORMAT CHECK: LOANS

Column: loan_date
loan_date
YYYY-MM-DD                  4722
DD/MM/YYYY or MM/DD/YYYY     995
DD Mon YYYY                  343
Name: count, dtype: int64

Examples:
['2025-01-01' '01 Jan 2025' '2025-01-02' '02/01/2025' '2025-01-03'
 '03/01/2025' '03 Jan 2025' '2025-01-04' '04 Jan 2025' '04/01/2025']

Column: due_date
due_date
YYYY-MM-DD                  4722
DD/MM/YYYY or MM/DD/YYYY     995
DD Mon YYYY                  343
Name: count, dtype: int64

Examples:
['2025-01-22' '2025-01-08' '2025-01-15' '22 Jan 2025' '08 Jan 2025'
 '15 Jan 2025' '2025-01-09' '2025-01-23' '23/01/2025' '09/01/2025']

Column: return_date
return_date
YYYY-MM-DD        

Next, I'll move on to check:
- Uppercase and lowercase differences.
- Leading and trailing spaces.
- Different values that become identical after converting to lowercase and trimming spaces.
- How many distinct values exist before and after this temporary standardisation.

In [19]:

# TEXT CONSISTENCY CHECK — ALL DATASETS
# Inspection only: original datasets are NOT modified.

text_columns = {
    "branches": {
        "df": branches,
        "columns": ["branch_name", "area"]
    },
    "items": {
        "df": items,
        "columns": ["title", "classification"]
    },
    "members": {
        "df": members,
        "columns": ["age_band", "membership_type", "postcode"]
    },
    "loans": {
        "df": loans,
        "columns": ["loan_id", "member_id", "item_id", "branch_id"]
    }
}

for dataset_name, info in text_columns.items():

    df = info["df"]

    print("\n" + "=" * 65)
    print("TEXT CONSISTENCY CHECK:", dataset_name.upper())
    print("=" * 65)

    for column in info["columns"]:

        original = df[column].dropna().astype(str)

        # Temporary normalisation for inspection only
        normalised = original.str.strip().str.casefold()

        print("\nColumn:", column)
        print("-" * 50)

        print("Distinct original values:", original.nunique())
        print("Distinct values after trimming and case normalisation:",
              normalised.nunique())

        # Identify different original values that become identical
        # after trimming spaces and ignoring case
        variants = (
            pd.DataFrame({
                "original": original,
                "normalised": normalised
            })
            .groupby("normalised")["original"]
            .unique()
        )

        differences = variants[variants.apply(len) > 1]

        if len(differences) > 0:
            print("\nPotential case/spacing inconsistencies:")

            for standard_value, original_values in differences.head(15).items():
                print(f"  {standard_value!r}: {list(original_values)}")

            if len(differences) > 15:
                print("  ... and", len(differences) - 15,
                      "more groups")
        else:
            print("\nNo case/spacing variations detected.")

        # Show original category counts for classification
        if column == "classification":
            print("\nALL ORIGINAL CLASSIFICATION VALUES:")
            print(df[column].value_counts(dropna=False).to_string())

        # Show postcode examples
        if column == "postcode":
            print("\nPOSTCODE EXAMPLES:")
            print(original.unique()[:20])




TEXT CONSISTENCY CHECK: BRANCHES

Column: branch_name
--------------------------------------------------
Distinct original values: 8
Distinct values after trimming and case normalisation: 8

No case/spacing variations detected.

Column: area
--------------------------------------------------
Distinct original values: 4
Distinct values after trimming and case normalisation: 4

No case/spacing variations detected.

TEXT CONSISTENCY CHECK: ITEMS

Column: title
--------------------------------------------------
Distinct original values: 195
Distinct values after trimming and case normalisation: 195

No case/spacing variations detected.

Column: classification
--------------------------------------------------
Distinct original values: 30
Distinct values after trimming and case normalisation: 16

Potential case/spacing inconsistencies:
  'audiobook / fiction': ['Audiobook / Fiction', 'Audiobook / Fiction ', 'Audiobook / fiction', 'Audiobook / FICTION']
  'audiobook / non-fiction': ['Audiob

In a relational database, each table should have a primary key that uniquely identifies each record. For 'branches', the primary key is branch_id; For 'items', the primary key is item_id; For 'members', the primary key is member_id; For 'loans', the primary key is loan_id.

And now i gonna check four things:

- Whether primary keys contain missing values.
- Whether primary keys are unique.
- Whether entire rows are duplicated.
- Whether the same ID appears with different information.

In [20]:

# STEP 3: PRIMARY KEY AND DUPLICATE CHECKS
# This code only inspects the data. It does not modify anything.

datasets = {
    "branches": {
        "df": branches,
        "key": "branch_id"
    },
    "items": {
        "df": items,
        "key": "item_id"
    },
    "members": {
        "df": members,
        "key": "member_id"
    },
    "loans": {
        "df": loans,
        "key": "loan_id"
    }
}

for name, info in datasets.items():

    df = info["df"]
    key = info["key"]

    print("\n" + "=" * 60)
    print("PRIMARY KEY CHECK:", name.upper())
    print("=" * 60)

    # 1. Check missing primary keys
    print("Missing primary keys:", df[key].isna().sum())

    # 2. Check unique primary keys
    print("Total records:", len(df))
    print("Unique primary keys:", df[key].nunique())

    # 3. Check duplicate primary keys
    duplicate_keys = df[df.duplicated(subset=key, keep=False)]

    print("Rows with repeated primary keys:", len(duplicate_keys))
    print("Number of repeated key groups:",
          duplicate_keys[key].nunique())

    # 4. Check completely duplicated rows
    print("Completely duplicated rows:", df.duplicated().sum())

    # 5. Check repeated IDs with different information
    conflicting_groups = []

    for key_value, group in duplicate_keys.groupby(key):
        if len(group.drop_duplicates()) > 1:
            conflicting_groups.append(key_value)

    print("Repeated IDs with different records:",
          len(conflicting_groups))

    if conflicting_groups:
        print("\nExamples of repeated IDs with different records:")

        print(
            duplicate_keys[
                duplicate_keys[key].isin(conflicting_groups)
            ].head(15).to_string(index=False)
        )

    else:
        print("No conflicting records found.")



PRIMARY KEY CHECK: BRANCHES
Missing primary keys: 0
Total records: 8
Unique primary keys: 8
Rows with repeated primary keys: 0
Number of repeated key groups: 0
Completely duplicated rows: 0
Repeated IDs with different records: 0
No conflicting records found.

PRIMARY KEY CHECK: ITEMS
Missing primary keys: 0
Total records: 1200
Unique primary keys: 1200
Rows with repeated primary keys: 0
Number of repeated key groups: 0
Completely duplicated rows: 0
Repeated IDs with different records: 0
No conflicting records found.

PRIMARY KEY CHECK: MEMBERS
Missing primary keys: 0
Total records: 832
Unique primary keys: 800
Rows with repeated primary keys: 64
Number of repeated key groups: 32
Completely duplicated rows: 24
Repeated IDs with different records: 8

Examples of repeated IDs with different records:
member_id  join_date age_band membership_type home_branch_id postcode
   M00017 2023-01-01 Under 12          Junior            B03  ZZ8 1YJ
   M00017 01/01/2023 Under 12          Junior      

As u can see, there are duplicated keys in loans and members.

In a relational database, foreign keys connect different tables. Now I will check all these relationships without modifying the data.

In [21]:

# STEP 4: REFERENTIAL INTEGRITY CHECK
# Inspection only - no records are modified.

relationships = [
    {
        "child": items,
        "column": "home_branch_id",
        "parent": branches,
        "parent_column": "branch_id",
        "relationship": "Items -> Branches"
    },
    {
        "child": members,
        "column": "home_branch_id",
        "parent": branches,
        "parent_column": "branch_id",
        "relationship": "Members -> Branches"
    },
    {
        "child": loans,
        "column": "member_id",
        "parent": members,
        "parent_column": "member_id",
        "relationship": "Loans -> Members"
    },
    {
        "child": loans,
        "column": "item_id",
        "parent": items,
        "parent_column": "item_id",
        "relationship": "Loans -> Items"
    },
    {
        "child": loans,
        "column": "branch_id",
        "parent": branches,
        "parent_column": "branch_id",
        "relationship": "Loans -> Branches"
    }
]

for relation in relationships:

    child = relation["child"]
    parent = relation["parent"]

    child_column = relation["column"]
    parent_column = relation["parent_column"]

    # Compare IDs after trimming spaces and standardising case
    child_ids = child[child_column].dropna().astype(str).str.strip().str.upper()
    parent_ids = set(
        parent[parent_column].dropna().astype(str).str.strip().str.upper()
    )

    unmatched = child_ids[~child_ids.isin(parent_ids)]

    print("\n" + "=" * 60)
    print(relation["relationship"])
    print("=" * 60)

    print("Records checked:", len(child_ids))
    print("Unmatched foreign keys:", len(unmatched))
    print("Distinct unmatched IDs:", unmatched.nunique())

    if len(unmatched) > 0:
        print("\nUnmatched ID examples:")
        print(unmatched.value_counts().head(15).to_string())
    else:
        print("All foreign keys have matching parent records.")



Items -> Branches
Records checked: 1200
Unmatched foreign keys: 0
Distinct unmatched IDs: 0
All foreign keys have matching parent records.

Members -> Branches
Records checked: 832
Unmatched foreign keys: 0
Distinct unmatched IDs: 0
All foreign keys have matching parent records.

Loans -> Members
Records checked: 6060
Unmatched foreign keys: 60
Distinct unmatched IDs: 42

Unmatched ID examples:
member_id
M00945    3
M00985    3
M00939    2
M00938    2
M00963    2
M00978    2
M00957    2
M00976    2
M00922    2
M00989    2
M00915    2
M00901    2
M00971    2
M00921    2
M00906    2

Loans -> Items
Records checked: 6060
Unmatched foreign keys: 30
Distinct unmatched IDs: 25

Unmatched ID examples:
item_id
I1212    2
I1205    2
I1224    2
I1266    2
I1290    2
I1248    1
I1209    1
I1295    1
I1211    1
I1251    1
I1272    1
I1278    1
I1265    1
I1222    1
I1282    1

Loans -> Branches
Records checked: 6060
Unmatched foreign keys: 0
Distinct unmatched IDs: 0
All foreign keys have matchin

From the result, relationship in Loans->Members and Loans->Items have some unmatched records. That means 60 loan records refer to 42 member IDs that aren't in the members table, and 30 loan records refer to 25 item IDs that aren't in the items table.

Now I'll inspect the numerical columns in my datasets. I'll also check whether the numerical columns have been loaded as the correct data types.

In [22]:

# STEP 5: NUMERICAL DATA QUALITY CHECK
# Inspection only - original datasets remain unchanged.

import pandas as pd

numerical_columns = {
    "branches": {
        "df": branches,
        "columns": ["opened_year"]
    },
    "items": {
        "df": items,
        "columns": ["replacement_cost"]
    },
    "loans": {
        "df": loans,
        "columns": ["fine_charged"]
    }
}

for dataset_name, info in numerical_columns.items():

    df = info["df"]

    print("\n" + "=" * 60)
    print("NUMERICAL DATA CHECK:", dataset_name.upper())
    print("=" * 60)

    for column in info["columns"]:

        # Create a temporary numeric version for inspection
        values = df[column].astype("string").str.replace(
            "£", "", regex=False
        ).str.strip()

        numeric_values = pd.to_numeric(values, errors="coerce")

        print("\nColumn:", column)
        print("Original data type:", df[column].dtype)
        print("Missing values:", numeric_values.isna().sum())
        print("Minimum:", numeric_values.min())
        print("Maximum:", numeric_values.max())
        print("Mean:", round(numeric_values.mean(), 2))
        print("Median:", numeric_values.median())
        print("Negative values:", (numeric_values < 0).sum())
        print("Zero values:", (numeric_values == 0).sum())

        print("\nFive smallest values:")
        print(numeric_values.nsmallest(5).to_list())

        print("\nFive largest values:")
        print(numeric_values.nlargest(5).to_list())

        # Check potential outliers using the IQR method
        q1 = numeric_values.quantile(0.25)
        q3 = numeric_values.quantile(0.75)
        iqr = q3 - q1

        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr

        outliers = numeric_values[
            (numeric_values < lower_limit) |
            (numeric_values > upper_limit)
        ]

        print("\nPotential outliers:", outliers.count())
        print("IQR lower limit:", round(lower_limit, 2))
        print("IQR upper limit:", round(upper_limit, 2))



NUMERICAL DATA CHECK: BRANCHES

Column: opened_year
Original data type: int64
Missing values: 0
Minimum: 1958
Maximum: 2011
Mean: 1982.75
Median: 1981.0
Negative values: 0
Zero values: 0

Five smallest values:
[1958, 1962, 1971, 1974, 1988]

Five largest values:
[2011, 2003, 1995, 1988, 1974]

Potential outliers: 0
IQR lower limit: 1926.38
IQR upper limit: 2039.38

NUMERICAL DATA CHECK: ITEMS

Column: replacement_cost
Original data type: object
Missing values: 24
Minimum: 6.0
Maximum: 29.92
Mean: 15.69
Median: 15.455
Negative values: 0
Zero values: 0

Five smallest values:
[6.0, 6.01, 6.02, 6.03, 6.09]

Five largest values:
[29.92, 29.71, 29.63, 29.58, 29.51]

Potential outliers: 0
IQR lower limit: -2.23
IQR upper limit: 33.25

NUMERICAL DATA CHECK: LOANS

Column: fine_charged
Original data type: object
Missing values: 773
Minimum: -6.0
Maximum: 100.0
Mean: 0.32
Median: 0.0
Negative values: 6
Zero values: 4499

Five smallest values:
[-6.0, -2.2, -2.0, -0.6, -0.2]

Five largest values:

Now, I want to establish whether the loan records make logical sense.

For a library loan, the expected rules are:
1. loan_date should not be later than due_date.
2. If an item has been returned, return_date should not be earlier than loan_date.
3. If an item has been returned, the return date should be checked against its due date.
4. If an item has not been returned, return_date should be missing.
5. A member should have joined the library on or before the loan date.

I will also investigate whether missing fine values are associated with unreturned loans.

In [23]:

# STEP 6: BUSINESS RULE AND DATE LOGIC CHECKS
# Inspection only - original data is not modified.

import pandas as pd

# Create a temporary copy of the loans data
loan_check = loans.copy()

# Convert dates using day-first interpretation
date_columns = ["loan_date", "due_date", "return_date"]

for column in date_columns:
    loan_check[column] = pd.to_datetime(
        loan_check[column],
        dayfirst=True,
        errors="coerce"
    )

# Prepare a temporary copy of members
member_check = members[["member_id", "join_date"]].copy()

member_check["join_date"] = pd.to_datetime(
    member_check["join_date"],
    dayfirst=True,
    errors="coerce"
)

# Attach member join dates to loan records
loan_check = loan_check.merge(
    member_check.drop_duplicates("member_id"),
    on="member_id",
    how="left"
)

print("=" * 60)
print("STEP 6: BUSINESS RULE VALIDATION")
print("=" * 60)

# Rule 1: Loan date must not be after due date
rule1 = loan_check["loan_date"] > loan_check["due_date"]

print("\n1. Loan date later than due date:")
print("Violations:", rule1.sum())

# Rule 2: Return date must not be before loan date
rule2 = loan_check["return_date"] < loan_check["loan_date"]

print("\n2. Return date earlier than loan date:")
print("Violations:", rule2.sum())

# Rule 3: Returned after due date
rule3 = loan_check["return_date"] > loan_check["due_date"]

print("\n3. Items returned after due date:")
print("Records:", rule3.sum())

# Rule 4: Missing return dates
print("\n4. Loans without a return date:")
print(loan_check["return_date"].isna().sum())

# Rule 5: Member join date after loan date
rule5 = loan_check["join_date"] > loan_check["loan_date"]

print("\n5. Member joined after the loan date:")
print("Violations:", rule5.sum())

# Rule 6: Missing fines by return status
print("\n6. Missing fines by return status:")

loan_check["return_status"] = loan_check["return_date"].apply(
    lambda x: "Not returned" if pd.isna(x) else "Returned"
)

print(
    loan_check.groupby("return_status")["fine_charged"]
    .agg(
        total_records="count",
        missing_fines=lambda x: x.isna().sum()
    )
)

# Show examples of date violations
print("\n7. Examples of return dates before loan dates:")

print(
    loan_check.loc[
        rule2,
        ["loan_id", "member_id", "loan_date",
         "due_date", "return_date"]
    ].head(10).to_string(index=False)
)

print("\n8. Examples of members joining after loan dates:")

print(
    loan_check.loc[
        rule5,
        ["loan_id", "member_id", "join_date", "loan_date"]
    ].head(10).to_string(index=False)
)


STEP 6: BUSINESS RULE VALIDATION

1. Loan date later than due date:
Violations: 867

2. Return date earlier than loan date:
Violations: 108

3. Items returned after due date:
Records: 768

4. Loans without a return date:
4322

5. Member joined after the loan date:
Violations: 25

6. Missing fines by return status:
               total_records  missing_fines
return_status                              
Not returned            3723            599
Returned                1564            174

7. Examples of return dates before loan dates:
loan_id member_id  loan_date   due_date return_date
L000056    M00200 2025-04-01 2025-01-25  2025-01-02
L000086    M00180 2025-06-01 2025-01-27  2025-01-02
L000152    M00205 2025-10-01 2025-01-31  2025-09-02
L000153    M00567 2025-10-01 2025-01-31  2025-09-02
L000159    M00655 2025-10-01 2025-01-31  2025-02-02
L000191    M00421 2025-12-01 2025-02-02  2025-01-02
L000192    M00096 2025-12-01 2025-02-02  2025-02-02
L000577    M00210 2025-05-02 2025-02-26  202

/tmp/ipykernel_1527/2619010422.py:13: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  loan_check[column] = pd.to_datetime(


I would not clean anything yet — first, we need to check whether the date results are genuine errors or caused by inconsistent date interpretation.

There is one important warning in your output:

dayfirst=True was applied to dates that are already in YYYY-MM-DD format. This can produce confusing results when the dataset contains several date formats. For example, a date such as 2025-04-01 should mean 1 April 2025, not 4 January 2025.

That may explain some of the 867 loan-date violations and 108 return-date violations. Let's verify the date parsing before drawing conclusions.

I'll parse each date according to its format:
- YYYY-MM-DD as year-month-day.
- Slash-separated dates as day-month-year, provisionally.
- Dates such as 02 Jan 2025 using their written month.

In [24]:
# STEP 7: CHECK DATE PARSING
# Inspection only - original data is not modified.

import pandas as pd

def parse_mixed_dates(series):
    def parse_one(value):
        if pd.isna(value):
            return pd.NaT

        value = str(value).strip()

        # ISO format: YYYY-MM-DD
        if len(value) >= 10 and value[4] == "-" and value[7] == "-":
            return pd.to_datetime(
                value,
                format="%Y-%m-%d",
                errors="coerce"
            )

        # Written month format, e.g. 02 Jan 2025
        if any(char.isalpha() for char in value):
            return pd.to_datetime(
                value,
                format="%d %b %Y",
                errors="coerce"
            )

        # Slash format: provisionally DD/MM/YYYY
        if "/" in value:
            return pd.to_datetime(
                value,
                format="%d/%m/%Y",
                errors="coerce"
            )

        return pd.NaT

    return series.apply(parse_one)


# Work on temporary copies
loan_check2 = loans.copy()
member_check2 = members.copy()

date_columns = ["loan_date", "due_date", "return_date"]

for column in date_columns:
    loan_check2[column] = parse_mixed_dates(loan_check2[column])

member_check2["join_date"] = parse_mixed_dates(
    member_check2["join_date"]
)

# Check how many dates could not be parsed
print("DATE PARSING CHECK")
print("=" * 50)

for column in date_columns:
    print(
        f"{column}:",
        loan_check2[column].isna().sum(),
        "missing or unparsed"
    )

print(
    "join_date:",
    member_check2["join_date"].isna().sum(),
    "missing or unparsed"
)

# Join member dates to loans
# Keep one row per member ID for this temporary check
member_dates = member_check2[
    ["member_id", "join_date"]
].drop_duplicates("member_id")

loan_check2 = loan_check2.merge(
    member_dates,
    on="member_id",
    how="left"
)

# Repeat the business-rule checks
print("\nBUSINESS RULE CHECKS AFTER DATE PARSING")
print("=" * 50)

print(
    "Loan date later than due date:",
    (loan_check2["loan_date"] > loan_check2["due_date"]).sum()
)

print(
    "Return date earlier than loan date:",
    (loan_check2["return_date"] < loan_check2["loan_date"]).sum()
)

print(
    "Returned after due date:",
    (loan_check2["return_date"] > loan_check2["due_date"]).sum()
)

print(
    "Member joined after loan date:",
    (loan_check2["join_date"] > loan_check2["loan_date"]).sum()
)

print(
    "Loans with no matched member join date:",
    loan_check2["join_date"].isna().sum()
)

DATE PARSING CHECK
loan_date: 0 missing or unparsed
due_date: 0 missing or unparsed
return_date: 247 missing or unparsed
join_date: 0 missing or unparsed

BUSINESS RULE CHECKS AFTER DATE PARSING
Loan date later than due date: 0
Return date earlier than loan date: 12
Returned after due date: 943
Member joined after loan date: 170
Loans with no matched member join date: 60


The earlier 867 loan-date violations have dropped to zero, which confirms that the previous date-parsing method was producing misleading results.

One important point: 943 late returns are not automatically errors. They may be legitimate overdue loans. Similarly, the 170 membership-date violations need investigation before we decide what to do with them.

So now, look at the actual records behind the 12 returns before loan dates and the 170 loans where the member appears to have joined after borrowing.

In [25]:
# STEP 8: INVESTIGATE DATE VIOLATIONS
# Inspection only - original data is not modified.

print("=" * 60)
print("1. RETURNS BEFORE LOAN DATES")
print("=" * 60)

return_violations = loan_check2[
    loan_check2["return_date"] < loan_check2["loan_date"]
]

print("Number of affected records:", len(return_violations))

print(
    return_violations[
        ["loan_id", "member_id", "loan_date",
         "due_date", "return_date"]
    ].sort_values("loan_date").to_string(index=False)
)


print("\n" + "=" * 60)
print("2. MEMBERS JOINING AFTER LOAN DATES")
print("=" * 60)

join_violations = loan_check2[
    loan_check2["join_date"] > loan_check2["loan_date"]
]

print("Number of affected records:", len(join_violations))

print(
    join_violations[
        ["loan_id", "member_id", "join_date", "loan_date"]
    ].sort_values("member_id").head(25).to_string(index=False)
)


print("\n" + "=" * 60)
print("3. HOW MANY DISTINCT MEMBERS ARE AFFECTED?")
print("=" * 60)

print(
    "Distinct members with join-date violations:",
    join_violations["member_id"].nunique()
)

print(
    "Distinct members with unmatched IDs:",
    loan_check2.loc[
        loan_check2["join_date"].isna(),
        "member_id"
    ].nunique()
)

1. RETURNS BEFORE LOAN DATES
Number of affected records: 12
loan_id member_id  loan_date   due_date return_date
L000343    M00065 2025-01-21 2025-02-11  2025-01-19
L000472    M00779 2025-01-30 2025-02-20  2025-01-22
L000648    M00412 2025-02-09 2025-03-02  2025-02-05
L000811    M00461 2025-02-19 2025-02-26  2025-02-12
L000865    M00559 2025-02-22 2025-03-15  2025-02-15
L000877    M00016 2025-02-23 2025-03-16  2025-02-15
L001125    M00234 2025-03-12 2025-03-26  2025-03-09
L001447    M00440 2025-04-01 2025-04-22  2025-03-23
L001703    M00616 2025-04-17 2025-05-01  2025-04-14
L002731    M00246 2025-06-23 2025-06-30  2025-06-20
L002997    M00340 2025-07-07 2025-07-28  2025-07-03
L004706    M00209 2025-09-29 2025-10-06  2025-09-19

2. MEMBERS JOINING AFTER LOAN DATES
Number of affected records: 170
loan_id member_id  join_date  loan_date
L002465    M00036 2025-06-12 2025-06-06
L000399    M00082 2025-03-24 2025-01-24
L000192    M00096 2025-06-15 2025-01-12
L000603    M00096 2025-06-15 2025-0

As the result, 12 loans have return dates before their loan dates. These are genuine date-logic violations and should be flagged for investigation.

170 loan records involve 45 members whose recorded join dates are after the loan dates. This may indicate incorrect member dates or incorrect loan records.

However, before we conclude that all 170 are errors, we need to address something we discovered earlier: the members table contains duplicate member IDs. Some repeated records have different date and postcode formatting. We should check whether these are actually different member records or simply formatting differences.

So I'll temporarily standardise the member fields and check whether repeated member IDs still have conflicting information.

In [26]:
# STEP 9: INVESTIGATE DUPLICATE MEMBER RECORDS
# Inspection only - original data is not modified.

member_duplicates_check = member_check2.copy()

# Standardise text fields temporarily
text_columns = [
    "age_band",
    "membership_type",
    "home_branch_id",
    "postcode"
]

for column in text_columns:
    member_duplicates_check[column] = (
        member_duplicates_check[column]
        .astype("string")
        .str.strip()
    )

# Standardise case
for column in ["age_band", "membership_type",
               "home_branch_id", "postcode"]:
    member_duplicates_check[column] = (
        member_duplicates_check[column].str.upper()
    )

# Remove spaces from postcodes for comparison
member_duplicates_check["postcode"] = (
    member_duplicates_check["postcode"].str.replace(
        " ", "", regex=False
    )
)

# Count distinct values per member ID after standardisation
comparison = member_duplicates_check.groupby("member_id").agg(
    record_count=("member_id", "size"),
    different_join_dates=("join_date", "nunique"),
    different_age_bands=("age_band", "nunique"),
    different_membership_types=("membership_type", "nunique"),
    different_home_branches=("home_branch_id", "nunique"),
    different_postcodes=("postcode", "nunique")
)

# Show member IDs that still have conflicting information
conflicts = comparison[
    (comparison["record_count"] > 1) &
    (
        (comparison["different_join_dates"] > 1) |
        (comparison["different_age_bands"] > 1) |
        (comparison["different_membership_types"] > 1) |
        (comparison["different_home_branches"] > 1) |
        (comparison["different_postcodes"] > 1)
    )
]

print("DUPLICATE MEMBER INVESTIGATION")
print("=" * 60)

print("Member IDs appearing more than once:",
      (comparison["record_count"] > 1).sum())

print("Member IDs with conflicting information after standardisation:",
      len(conflicts))

print("\nConflicting member records:")
print(conflicts.to_string())

print("\nRepeated IDs that become identical after standardisation:")
print(
    (
        (comparison["record_count"] > 1) &
        (comparison["different_join_dates"] == 1) &
        (comparison["different_age_bands"] == 1) &
        (comparison["different_membership_types"] == 1) &
        (comparison["different_home_branches"] == 1) &
        (comparison["different_postcodes"] == 1)
    ).sum()
)

DUPLICATE MEMBER INVESTIGATION
Member IDs appearing more than once: 32
Member IDs with conflicting information after standardisation: 0

Conflicting member records:
Empty DataFrame
Columns: [record_count, different_join_dates, different_age_bands, different_membership_types, different_home_branches, different_postcodes]
Index: []

Repeated IDs that become identical after standardisation:
30


It suggests that the repeated member IDs are mostly formatting duplicates rather than genuinely different member records.
- 32 member IDs appear more than once.
- 30 repeated IDs become identical after standardisation.
- No conflicts were detected by the check.

There is one small issue with our previous code: Pandas' nunique() ignores missing values by default. This means a missing postcode and a populated postcode could be treated as if they were identical. Let's check those two remaining records properly before moving on.

Count missing values as a distinct value when comparing repeated member IDs. This will tell us whether any duplicate records differ because one version has missing information.

In [27]:
# STEP 10: FINAL DUPLICATE MEMBER CHECK
# Inspection only - original data is not modified.

comparison_final = member_duplicates_check.groupby("member_id").agg(
    record_count=("member_id", "size"),
    different_join_dates=("join_date", lambda x: x.nunique(dropna=False)),
    different_age_bands=("age_band", lambda x: x.nunique(dropna=False)),
    different_membership_types=("membership_type", lambda x: x.nunique(dropna=False)),
    different_home_branches=("home_branch_id", lambda x: x.nunique(dropna=False)),
    different_postcodes=("postcode", lambda x: x.nunique(dropna=False))
)

# Find repeated IDs with any differences, including missing values
duplicate_conflicts = comparison_final[
    (comparison_final["record_count"] > 1) &
    (
        (comparison_final["different_join_dates"] > 1) |
        (comparison_final["different_age_bands"] > 1) |
        (comparison_final["different_membership_types"] > 1) |
        (comparison_final["different_home_branches"] > 1) |
        (comparison_final["different_postcodes"] > 1)
    )
]

print("FINAL DUPLICATE MEMBER CHECK")
print("=" * 60)

print("Repeated member IDs:", (comparison_final["record_count"] > 1).sum())
print("Repeated IDs with differences:", len(duplicate_conflicts))

print("\nDetails:")
print(duplicate_conflicts.to_string())

# Show the original records for any remaining differences
if len(duplicate_conflicts) > 0:
    print("\nOriginal records for these member IDs:")

    print(
        member_duplicates_check[
            member_duplicates_check["member_id"].isin(
                duplicate_conflicts.index
            )
        ].sort_values("member_id").to_string(index=False)
    )

FINAL DUPLICATE MEMBER CHECK
Repeated member IDs: 32
Repeated IDs with differences: 0

Details:
Empty DataFrame
Columns: [record_count, different_join_dates, different_age_bands, different_membership_types, different_home_branches, different_postcodes]
Index: []


Oh! I forgot to check whether the postcode is correctly formatted. So let's check this now before moving on. It's a worthwhile part of data profiling.

In [28]:

# STEP 11: POSTCODE FORMAT VALIDATION
# Inspection only - original data is not modified.

postcode_check = members["postcode"].astype("string")

# Create temporary standardised versions
postcode_trimmed = postcode_check.str.strip()
postcode_upper = postcode_trimmed.str.upper()

# Remove whitespace for checking the underlying postcode characters
postcode_no_spaces = postcode_upper.str.replace(
    r"\s+", "", regex=True
)

# Expected synthetic postcode format after removing spaces:
# ZZ + digit + digit + two letters
valid_pattern = postcode_no_spaces.str.fullmatch(
    r"ZZ\d{2}[A-Z]{2}", na=False
)

print("=" * 60)
print("POSTCODE QUALITY CHECK")
print("=" * 60)

# 1. Missing values
print("\n1. Missing postcodes:")
print(postcode_check.isna().sum())

# 2. Leading or trailing spaces
print("\n2. Values with leading/trailing spaces:")
print(
    (
        postcode_check.notna() &
        (postcode_check != postcode_trimmed)
    ).sum()
)

# 3. Lowercase letters
print("\n3. Values containing lowercase letters:")
print(
    (
        postcode_check.notna() &
        (postcode_check != postcode_upper)
    ).sum()
)

# 4. Internal spacing problems
print("\n4. Values with incorrect spacing:")

expected_spacing = postcode_upper.str.fullmatch(
    r"ZZ\d \d[A-Z]{2}", na=False
)

print(
    (
        postcode_check.notna() &
        ~expected_spacing
    ).sum()
)

# 5. Format validation
print("\n5. Postcode format validation:")

print("Valid format:", valid_pattern.sum())

print(
    "Invalid format:",
    (postcode_check.notna() & ~valid_pattern).sum()
)

# 6. Display examples of invalid values
print("\n6. Examples of invalid postcodes:")

invalid_postcodes = postcode_check[
    postcode_check.notna() & ~valid_pattern
]

print(invalid_postcodes.head(20).to_string(index=False))

# 7. Show examples of values needing formatting corrections
print("\n7. Examples of formatting differences:")

formatting_issues = postcode_check[
    postcode_check.notna() &
    (
        (postcode_check != postcode_upper) |
        (postcode_check != postcode_trimmed) |
        ~expected_spacing
    )
]

print(formatting_issues.head(20).to_string(index=False))


POSTCODE QUALITY CHECK

1. Missing postcodes:
34

2. Values with leading/trailing spaces:
0

3. Values containing lowercase letters:
128

4. Values with incorrect spacing:
82

5. Postcode format validation:
Valid format: 798
Invalid format: 0

6. Examples of invalid postcodes:
Series([], )

7. Examples of formatting differences:
 ZZ86RB
zz8 1yj
 ZZ42UP
zz4 9gp
 ZZ37LW
zz2 5wg
zz6 2fq
 zz99ur
zz9 1xx
zz5 9ej
 zz69pt
 zz69pt
zz9 5dz
 ZZ26SN
zz4 1ue
 ZZ27JT
 ZZ58BF
zz8 3uh
 ZZ72LT
zz1 8uh


The last result means all 798 non-missing values have the expected characters when we ignore capitalisation and spaces. It does not mean all 798 are already correctly formatted. We'll standardise those later in Silver.

I would do one final check before moving to Silver: validate whether loan periods and fines agree with the library's business rules.

In [29]:

# STEP 12: LOAN PERIOD AND FINE VALIDATION
# Inspection only - original data is not modified.

# Remove exact duplicate loan rows for this diagnostic check
loan_rules = loans.drop_duplicates().copy()

# Parse dates using the formats found in the source files
for column in ["loan_date", "due_date", "return_date"]:
    loan_rules[column] = parse_mixed_dates(loan_rules[column])

# Prepare item information
item_rules = items[["item_id", "classification"]].copy()

# Split the combined classification into item type and category
split_classification = (
    item_rules["classification"]
    .astype("string")
    .str.split("/", n=1, expand=True)
)

item_rules["item_type"] = (
    split_classification[0].str.strip().str.upper()
)

item_rules["category"] = (
    split_classification[1].str.strip().str.lower()
)

# Standardise category names for this diagnostic check
category_mapping = {
    "fiction": "Fiction",
    "non-fiction": "Non-fiction",
    "non fiction": "Non-fiction",
    "nonfiction": "Non-fiction",
    "children": "Children",
    "childrens": "Children",
    "children's": "Children",
    "reference": "Reference",
    "ref.": "Reference",
    "film": "Film",
    "films": "Film"
}

item_rules["category"] = item_rules["category"].map(
    category_mapping
)

# Standardise item type
type_mapping = {
    "BOOK": "Book",
    "DVD": "DVD",
    "AUDIOBOOK": "Audiobook"
}

item_rules["item_type"] = item_rules["item_type"].map(
    type_mapping
)

# Join item information to loans
loan_rules = loan_rules.merge(
    item_rules,
    on="item_id",
    how="left"
)

print("=" * 60)
print("LOAN PERIOD VALIDATION")
print("=" * 60)

# Expected loan periods
expected_periods = {
    "Book": 21,
    "DVD": 7,
    "Audiobook": 14
}

loan_rules["actual_loan_days"] = (
    loan_rules["due_date"] - loan_rules["loan_date"]
).dt.days

loan_rules["expected_loan_days"] = (
    loan_rules["item_type"].map(expected_periods)
)

period_issues = loan_rules[
    loan_rules["actual_loan_days"].notna() &
    loan_rules["expected_loan_days"].notna() &
    (
        loan_rules["actual_loan_days"] !=
        loan_rules["expected_loan_days"]
    )
]

print("Loans with incorrect loan periods:", len(period_issues))

print("\nExamples:")
print(
    period_issues[
        ["loan_id", "item_type", "loan_date",
         "due_date", "actual_loan_days", "expected_loan_days"]
    ].head(10).to_string(index=False)
)

# Check classification parsing
print("\nClassification values not successfully mapped:")
print(
    loan_rules[
        loan_rules["item_type"].isna() |
        loan_rules["category"].isna()
    ]["item_id"].nunique()
)

print("\n" + "=" * 60)
print("FINE VALIDATION")
print("=" * 60)

# Convert fine amounts into numeric values
loan_rules["fine_numeric"] = pd.to_numeric(
    loan_rules["fine_charged"]
    .astype("string")
    .str.replace("£", "", regex=False)
    .str.strip(),
    errors="coerce"
)

# Only use returned loans with valid date order
valid_returns = loan_rules[
    loan_rules["return_date"].notna() &
    (loan_rules["return_date"] >= loan_rules["loan_date"]) &
    loan_rules["category"].notna() &
    loan_rules["item_type"].notna()
].copy()

# Calculate overdue days
valid_returns["late_days"] = (
    valid_returns["return_date"] -
    valid_returns["due_date"]
).dt.days.clip(lower=0)

# Calculate expected fines
fine_rates = {
    "Book": 0.20,
    "DVD": 1.00,
    "Audiobook": 0.20
}

valid_returns["fine_rate"] = (
    valid_returns["item_type"].map(fine_rates)
)

valid_returns["expected_fine"] = (
    valid_returns["late_days"] *
    valid_returns["fine_rate"]
).round(2).clip(upper=10)

# Children-category items have no fines
valid_returns.loc[
    valid_returns["category"] == "Children",
    "expected_fine"
] = 0

# Compare actual and expected fines
fine_mismatches = valid_returns[
    valid_returns["fine_numeric"].notna() &
    (
        (valid_returns["fine_numeric"] -
         valid_returns["expected_fine"]).abs() > 0.001
    )
]

missing_positive_fines = valid_returns[
    valid_returns["fine_numeric"].isna() &
    (valid_returns["expected_fine"] > 0)
]

missing_zero_fines = valid_returns[
    valid_returns["fine_numeric"].isna() &
    (valid_returns["expected_fine"] == 0)
]

print("Returned loans checked:", len(valid_returns))
print("Fines differing from expected:", len(fine_mismatches))
print("Missing fines where a positive fine was expected:",
      len(missing_positive_fines))
print("Missing fines where zero was expected:",
      len(missing_zero_fines))

print("\nExamples of fine mismatches:")
print(
    fine_mismatches[
        ["loan_id", "item_type", "category", "late_days",
         "fine_numeric", "expected_fine"]
    ].head(10).to_string(index=False)
)

print("\nFines greater than £10:",
      (loan_rules["fine_numeric"] > 10).sum())

print("Negative fines:",
      (loan_rules["fine_numeric"] < 0).sum())


LOAN PERIOD VALIDATION
Loans with incorrect loan periods: 0

Examples:
Empty DataFrame
Columns: [loan_id, item_type, loan_date, due_date, actual_loan_days, expected_loan_days]
Index: []

Classification values not successfully mapped:
25

FINE VALIDATION
Returned loans checked: 5713
Fines differing from expected: 9
Missing fines where a positive fine was expected: 0
Missing fines where zero was expected: 517

Examples of fine mismatches:
loan_id item_type    category  late_days  fine_numeric  expected_fine
L000008       DVD        Film          6          -6.0            6.0
L000074      Book     Fiction         11          -2.2            2.2
L000926      Book     Fiction         10          -2.0            2.0
L001064      Book   Reference          1          -0.2            0.2
L002322      Book Non-fiction          1          20.0            0.2
L003372      Book Non-fiction          1          20.0            0.2
L004268       DVD        Film          1         100.0            1.0

In [30]:
# Investigate unmapped classification values

unmapped_items = item_rules[
    item_rules["item_type"].isna() |
    item_rules["category"].isna()
]

print("Number of unmapped items:")
print(len(unmapped_items))

print("\nOriginal classification values:")
print(
    unmapped_items["classification"]
    .value_counts(dropna=False)
)

Number of unmapped items:
0

Original classification values:
Series([], Name: count, dtype: int64)


In [31]:
print("Original dataset columns:")
print(df.columns.tolist())

print("\nMapping table columns:")
print(item_rules.columns.tolist())

Original dataset columns:
['loan_id', 'member_id', 'item_id', 'branch_id', 'loan_date', 'due_date', 'return_date', 'fine_charged']

Mapping table columns:
['item_id', 'classification', 'item_type', 'category']


This tells us something important: my df is the loan dataset, and it doesn't contain a classification column. The item_rules table contains the classification information and its mapped values.

My previous check found no missing item_type or category values in item_rules. Let's now check whether every item in the loan dataset has a corresponding entry in the mapping table.

In [32]:
# Check whether every loan item_id exists in item_rules
loan_item_ids = df["item_id"].dropna().unique()
mapped_item_ids = item_rules["item_id"].dropna().unique()

unmatched_item_ids = set(loan_item_ids) - set(mapped_item_ids)

print("Unique item IDs in loan dataset:", len(loan_item_ids))
print("Unique item IDs in mapping table:", len(mapped_item_ids))
print("Loan item IDs without a mapping:", len(unmatched_item_ids))

if unmatched_item_ids:
    print("\nUnmatched item IDs:")
    print(sorted(unmatched_item_ids))

Unique item IDs in loan dataset: 1152
Unique item IDs in mapping table: 1200
Loan item IDs without a mapping: 25

Unmatched item IDs:
['I1203', 'I1205', 'I1209', 'I1211', 'I1212', 'I1216', 'I1222', 'I1224', 'I1226', 'I1233', 'I1240', 'I1248', 'I1251', 'I1252', 'I1256', 'I1257', 'I1263', 'I1265', 'I1266', 'I1272', 'I1275', 'I1278', 'I1282', 'I1290', 'I1295']


Then check the classification mapping table itself:

In [33]:
print("Total mapping rows:", len(item_rules))
print("Missing classifications:", item_rules["classification"].isna().sum())
print("Missing item types:", item_rules["item_type"].isna().sum())
print("Missing categories:", item_rules["category"].isna().sum())

print("\nClassification values:")
print(item_rules["classification"].value_counts(dropna=False))

Total mapping rows: 1200
Missing classifications: 0
Missing item types: 0
Missing categories: 0

Classification values:
classification
Book / Fiction              306
Book / Non-fiction          247
Book / Children             200
DVD / Film                   98
Audiobook / Fiction          64
DVD / Children               61
Audiobook / Non-fiction      45
Book / Reference             35
Book / fiction               18
Book / Childrens             15
Book / FICTION               12
Book / children              12
Book / Fiction               11
Book / non-fiction           10
DVD / Films                   8
Book / nonfiction             7
Book / Non-Fiction            7
Book / Children's             7
Book / Ref.                   6
Audiobook / Fiction           6
DVD / film                    5
Book / Non fiction            5
Audiobook / Non-Fiction       3
DVD / Childrens               2
DVD / children                2
Audiobook / fiction           2
Audiobook / Non fiction       2
A

We can  find that 25 item IDs in the loan dataset do not have a matching record in item_rules.

Let's check whether the missing IDs are outside the range of IDs in the mapping table.

In [34]:
print("Mapping table item ID range:")
print("First:", item_rules["item_id"].min())
print("Last:", item_rules["item_id"].max())

print("\nLast 30 item IDs in mapping table:")
print(
    sorted(item_rules["item_id"].dropna().unique())[-30:]
)

print("\nUnmatched item ID range:")
print("First:", min(unmatched_item_ids))
print("Last:", max(unmatched_item_ids))

Mapping table item ID range:
First: I0001
Last: I1200

Last 30 item IDs in mapping table:
['I1171', 'I1172', 'I1173', 'I1174', 'I1175', 'I1176', 'I1177', 'I1178', 'I1179', 'I1180', 'I1181', 'I1182', 'I1183', 'I1184', 'I1185', 'I1186', 'I1187', 'I1188', 'I1189', 'I1190', 'I1191', 'I1192', 'I1193', 'I1194', 'I1195', 'I1196', 'I1197', 'I1198', 'I1199', 'I1200']

Unmatched item ID range:
First: I1203
Last: I1295


Check for duplicate IDs in the mapping table

In [35]:
duplicate_ids = item_rules[
    item_rules["item_id"].duplicated(keep=False)
]

print("Number of rows with duplicate item IDs:", len(duplicate_ids))
print(duplicate_ids.sort_values("item_id"))

Number of rows with duplicate item IDs: 0
Empty DataFrame
Columns: [item_id, classification, item_type, category]
Index: []


Mapping table contains IDs from I0001 to I1200, with no duplicates. However, the loan dataset references 25 IDs between I1203 and I1295 that aren't present in that table.

This strongly suggests that the loan data references items that were not included in the mapping table. It is a referential integrity issue, rather than missing classification values.

So now, i gonna find out how often the unmatched items are borrowed.

In [36]:
unmatched_loans = df[
    df["item_id"].isin(unmatched_item_ids)
]

print("Loan records involving unmatched items:")
print(len(unmatched_loans))

print("\nLoans per unmatched item:")
print(
    unmatched_loans["item_id"]
    .value_counts()
    .sort_index()
)

print("\nSample unmatched loan records:")
display(unmatched_loans.head(10))

Loan records involving unmatched items:
30

Loans per unmatched item:
item_id
I1203    1
I1205    2
I1209    1
I1211    1
I1212    2
I1216    1
I1222    1
I1224    2
I1226    1
I1233    1
I1240    1
I1248    1
I1251    1
I1252    1
I1256    1
I1257    1
I1263    1
I1265    1
I1266    2
I1272    1
I1275    1
I1278    1
I1282    1
I1290    2
I1295    1
Name: count, dtype: int64

Sample unmatched loan records:


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged
361,L000357,M00529,I1212,B08,2025-01-22,2025-02-12,2025-02-05,0.00
558,L000553,M00340,I1211,B06,2025-02-03,2025-02-24,2025-02-12,0.00
743,L000737,M00547,I1209,B07,2025-02-15,2025-03-08,2025-02-20,0.00
841,L000834,M00690,I1248,B01,2025-02-21,2025-03-14,2025-03-04,0.00
1095,L001087,M00198,I1212,b01,2025-03-10,2025-03-31,2025-03-12,0.00
1122,L001114,M00758,I1205,B03,12/03/2025,02/04/2025,26/03/2025,0.00
1352,L001344,M00581,I1295,B04,2025-03-26,2025-04-16,2025-04-20,0.80
1373,L001365,M00218,I1224,B01,2025-03-27,2025-04-17,2025-04-06,0.00
1403,L001394,M00483,I1266,B08,29/03/2025,19/04/2025,15/04/2025,0.00
1600,L001590,M00657,I1251,B07,10 Apr 2025,01 May 2025,21 Apr 2025,0.00


The important next step is to check whether the problem is in the original generated data or in how the mapping table item_rules was created.

In [37]:
print("Original items table:")
print("Shape:", items.shape)
print("Columns:", items.columns.tolist())

print("\nItem ID range:")
print("First:", items["item_id"].min())
print("Last:", items["item_id"].max())

print("\nNumber of unique item IDs:")
print(items["item_id"].nunique())

print("\nItems with IDs above I1200:")
print(
    items[items["item_id"] > "I1200"]
)

Original items table:
Shape: (1200, 5)
Columns: ['item_id', 'title', 'classification', 'replacement_cost', 'home_branch_id']

Item ID range:
First: I0001
Last: I1200

Number of unique item IDs:
1200

Items with IDs above I1200:
Empty DataFrame
Columns: [item_id, title, classification, replacement_cost, home_branch_id]
Index: []


Compare the original items table with the mapping table

In [38]:
original_item_ids = set(items["item_id"].dropna())
mapping_item_ids = set(item_rules["item_id"].dropna())

print("Items missing from mapping table:",
      len(original_item_ids - mapping_item_ids))

print("Mapping IDs not in original items table:",
      len(mapping_item_ids - original_item_ids))

print("\nIDs in original items but not mapping:")
print(sorted(original_item_ids - mapping_item_ids))

Items missing from mapping table: 0
Mapping IDs not in original items table: 0

IDs in original items but not mapping:
[]


Excellent. We've now isolated the issue: the original items table and the mapping table are consistent, but the loan dataset contains 25 item IDs that do not exist in either table.

So, next I will find out whether the loan records were corrupted

In [39]:
print("Available loan-related variables:")

for name in ["loans", "loan_data", "df"]:
    if name in globals():
        obj = globals()[name]
        if hasattr(obj, "columns"):
            print(
                name,
                "| Shape:", obj.shape,
                "| Columns:", obj.columns.tolist()
            )

Available loan-related variables:
loans | Shape: (6060, 8) | Columns: ['loan_id', 'member_id', 'item_id', 'branch_id', 'loan_date', 'due_date', 'return_date', 'fine_charged']
df | Shape: (6060, 8) | Columns: ['loan_id', 'member_id', 'item_id', 'branch_id', 'loan_date', 'due_date', 'return_date', 'fine_charged']


In [40]:
if "loans" in globals():
    original_unmatched = set(
        loans["item_id"].dropna()
    ) - set(items["item_id"].dropna())

    print("Unmatched IDs in original loans:", len(original_unmatched))
    print(sorted(original_unmatched))

Unmatched IDs in original loans: 25
['I1203', 'I1205', 'I1209', 'I1211', 'I1212', 'I1216', 'I1222', 'I1224', 'I1226', 'I1233', 'I1240', 'I1248', 'I1251', 'I1252', 'I1256', 'I1257', 'I1263', 'I1265', 'I1266', 'I1272', 'I1275', 'I1278', 'I1282', 'I1290', 'I1295']


The next step is to inspect how the generator creates loan records.

In [41]:
print("Maximum item ID in original items:")
print(items["item_id"].max())

print("\nMaximum item ID in original loans:")
print(loans["item_id"].max())

print("\nUnmatched loan records:")
display(
    loans[loans["item_id"].isin(original_unmatched)]
    .sort_values("item_id")
)

Maximum item ID in original items:
I1200

Maximum item ID in original loans:
I1295

Unmatched loan records:


,loan_id,member_id,item_id,branch_id,loan_date,due_date,return_date,fine_charged
1948,L001934,M00543,I1203,B03,03 May 2025,24 May 2025,05 May 2025,0.00
1965,L001951,M00022,I1205,B08,2025-05-04,2025-05-25,2025-05-23,NaN
1122,L001114,M00758,I1205,B03,12/03/2025,02/04/2025,26/03/2025,0.00
743,L000737,M00547,I1209,B07,2025-02-15,2025-03-08,2025-02-20,0.00
558,L000553,M00340,I1211,B06,2025-02-03,2025-02-24,2025-02-12,0.00
361,L000357,M00529,I1212,B08,2025-01-22,2025-02-12,2025-02-05,0.00
1095,L001087,M00198,I1212,b01,2025-03-10,2025-03-31,2025-03-12,0.00
3376,L003342,M00557,I1216,B04,2025-07-22,2025-08-12,2025-08-05,0.00
2150,L002133,M00242,I1222,B02,15/05/2025,05/06/2025,06/06/2025,0.20
1373,L001365,M00218,I1224,B01,2025-03-27,2025-04-17,2025-04-06,0.00


Next, let's verify the exact number of affected records

In [42]:
print("Total original loan records:", len(loans))

print(
    "Loan records with invalid item references:",
    loans["item_id"].isin(original_unmatched).sum()
)

print(
    "Distinct invalid item IDs:",
    loans.loc[
        loans["item_id"].isin(original_unmatched),
        "item_id"
    ].nunique()
)

Total original loan records: 6060
Loan records with invalid item references: 30
Distinct invalid item IDs: 25


## 3. Bronze layer – raw data ingestion

For now, let's create a Bronze directory and save the original DataFrames as CSV files.

In [43]:

from pathlib import Path

# Create Bronze directory
bronze_path = Path("data/bronze")
bronze_path.mkdir(parents=True, exist_ok=True)

# Original source datasets
bronze_tables = {
    "items": items,
    "loans": loans,
    "members": members,
    "branches": branches
}

# Save original datasets without modifying them
for table_name, table_df in bronze_tables.items():
    output_file = bronze_path / f"{table_name}.csv"
    table_df.to_csv(output_file, index=False)

    print(
        f"{table_name}: {len(table_df)} records saved to {output_file}"
    )


items: 1200 records saved to data/bronze/items.csv
loans: 6060 records saved to data/bronze/loans.csv
members: 832 records saved to data/bronze/members.csv
branches: 8 records saved to data/bronze/branches.csv


let's validate the bronze layer.

In [44]:

import pandas as pd

# Check the saved Bronze datasets
for table_name in bronze_tables:
    file_path = bronze_path / f"{table_name}.csv"
    bronze_df = pd.read_csv(file_path)

    print(f"\n{table_name.upper()}")
    print("Original records:", len(bronze_tables[table_name]))
    print("Bronze records:", len(bronze_df))
    print("Columns match:",
          list(bronze_tables[table_name].columns) == list(bronze_df.columns))



ITEMS
Original records: 1200
Bronze records: 1200
Columns match: True

LOANS
Original records: 6060
Bronze records: 6060
Columns match: True

MEMBERS
Original records: 832
Bronze records: 832
Columns match: True

BRANCHES
Original records: 8
Bronze records: 8
Columns match: True


I wanna identify all my available source datasets.

In [45]:
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")

df: (6060, 8)
branches: (8, 4)
items: (1200, 5)
loans: (6060, 8)
members: (832, 6)
duplicate_keys: (120, 8)
group: (2, 8)
child: (6060, 8)
parent: (8, 4)
loan_check: (6060, 10)
member_check: (832, 2)
loan_check2: (6060, 9)
member_check2: (832, 6)
member_dates: (800, 2)
return_violations: (12, 9)
join_violations: (170, 9)
member_duplicates_check: (832, 6)
comparison: (800, 6)
conflicts: (0, 6)
comparison_final: (800, 6)
duplicate_conflicts: (0, 6)
loan_rules: (6000, 14)
item_rules: (1200, 4)
split_classification: (1200, 2)
period_issues: (0, 13)
valid_returns: (5713, 17)
fine_mismatches: (9, 17)
missing_positive_fines: (0, 17)
missing_zero_fines: (517, 17)
unmapped_items: (0, 4)
duplicate_ids: (0, 4)
unmatched_loans: (30, 8)
obj: (6060, 8)
table_df: (8, 4)
bronze_df: (8, 4)


## 4. Silver layer — data cleaning and transformation

Now, create the silver folder first.

In [46]:
from pathlib import Path

silver_path = Path("data/silver")
silver_path.mkdir(parents=True, exist_ok=True)

print("Silver folder created:", silver_path)

Silver folder created: data/silver


From our earlier inspection, we already identified several issues that need investigating. We should not automatically delete or overwrite these records. We'll decide how to handle each issue and record the decisions.

We should not automatically delete or overwrite these records. I'll decide how to handle each issue and record the decisions.

Let's begin by creating a Silver copy of items.

In [47]:
silver_items = items.copy()

print("Original items:", items.shape)
print("Silver items:", silver_items.shape)
print("Missing values:")
print(silver_items.isnull().sum())

Original items: (1200, 5)
Silver items: (1200, 5)
Missing values:
item_id              0
title                0
classification       0
replacement_cost    24
home_branch_id       0
dtype: int64


We should investigate these missing costs before deciding how to handle them. We can not replace them with zero or the average yet, because that could introduce inaccurate financial information.

In [48]:
# Find items with missing replacement costs
missing_costs = silver_items[
    silver_items["replacement_cost"].isna()
]

print("Number of missing replacement costs:",
      len(missing_costs))

display(missing_costs)

Number of missing replacement costs: 24


,item_id,title,classification,replacement_cost,home_branch_id
26,I0027,The Secret Orchard,Book / fiction,NaN,B08
63,I0064,The Silver Promise,Book / Fiction,NaN,B07
65,I0066,The Hidden Engine,DVD / Children,NaN,B02
117,I0118,The Quiet Orchard,DVD / Film,NaN,B05
131,I0132,The Silver Forest,DVD / Film,NaN,B01
137,I0138,The Golden Stranger,Audiobook / Fiction,NaN,B05
221,I0222,The Quiet Lighthouse,Book / Fiction,NaN,B03
237,I0238,The Secret Kingdom,Book / Fiction,NaN,B01
335,I0336,The Quiet Orchard,DVD / Children,NaN,B01
378,I0379,The Distant Harbour,Book / Non-fiction,NaN,B06


In [50]:
# Understand the distribution of existing replacement costs
print(silver_items["replacement_cost"].describe())

# Check whether missing costs are concentrated in particular classifications
display(
    missing_costs["classification"].value_counts()
)

count      1176
unique      926
top       15.03
freq          4
Name: replacement_cost, dtype: object


,count
classification,
Book / Fiction,5
Book / Non-fiction,5
Audiobook / Fiction,3
DVD / Children,3
Book / Children,2
DVD / Film,2
Book / fiction,1
Audiobook / Non-fiction,1
Book / Reference,1


let's check whether the existing costs can safely be converted to numbers and whether any values fall outside the expected ranges.

In [51]:
# Check the current data types of replacement costs
print("Current data types:")
print(silver_items["replacement_cost"].map(type).value_counts())

# Convert costs to numeric in a separate column for investigation
cost_numeric = pd.to_numeric(
    silver_items["replacement_cost"],
    errors="coerce"
)

# Check whether any non-missing values failed conversion
invalid_costs = silver_items[
    silver_items["replacement_cost"].notna() &
    cost_numeric.isna()
]

print("\nValues that could not be converted:")
display(invalid_costs)

# Join item types from the reference mapping
items_with_type = silver_items.merge(
    item_rules[["item_id", "item_type"]],
    on="item_id",
    how="left"
)

items_with_type["replacement_cost_numeric"] = cost_numeric

# Summarise replacement costs by item type
print("\nReplacement cost summary by item type:")
display(
    items_with_type.groupby("item_type")[
        "replacement_cost_numeric"
    ].agg(["count", "min", "max", "median"])
)

Current data types:
replacement_cost
<class 'str'>      1176
<class 'float'>      24
Name: count, dtype: int64

Values that could not be converted:


,item_id,title,classification,replacement_cost,home_branch_id
0,I0001,The Secret Map,Book / Children,£18.98,B03
2,I0003,The Hidden River,Book / Non-fiction,£16.19,B01
5,I0006,The Hidden Lighthouse,Book / Non-fiction,£17.96,B07
10,I0011,The Broken Engine,Audiobook / Fiction,£26.45,B07
11,I0012,The Last Promise,Book / Children,£19.98,B04
...,...,...,...,...,...
1019,I1020,The Secret Season,DVD / Children,£9.76,B06
1099,I1100,The Paper Bridge,DVD / Film,£13.24,B02
1118,I1119,The Distant Bridge,Book / Non-fiction,£18.98,B07
1135,I1136,The Secret Map,Book / Fiction,£22.90,B01



Replacement cost summary by item type:


,count,min,max,median
item_type,,,,
Audiobook,113,12.18,29.92,20.23
Book,832,6.00,24.99,15.39
DVD,159,8.09,19.72,13.65


We've identified an important issue with the previous test. The 72 rows are not necessarily invalid costs. They failed conversion because the values contain a pound sign (£). So we need to remove the currency symbol before converting the column to numeric.

In [52]:
# Work on the Silver copy only
silver_items["replacement_cost"] = pd.to_numeric(
    silver_items["replacement_cost"]
        .astype("string")
        .str.replace("£", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
    errors="coerce"
)

# Check the results
print("Data type:")
print(silver_items["replacement_cost"].dtype)

print("\nMissing replacement costs:")
print(silver_items["replacement_cost"].isna().sum())

print("\nReplacement cost summary:")
print(silver_items["replacement_cost"].describe())

print("\nSample cleaned values:")
display(silver_items[["item_id", "replacement_cost"]].head(10))

Data type:
Float64

Missing replacement costs:
24

Replacement cost summary:
count       1176.0
mean     15.694796
std       5.519035
min            6.0
25%         11.075
50%         15.455
75%         19.945
max          29.92
Name: replacement_cost, dtype: Float64

Sample cleaned values:


,item_id,replacement_cost
0,I0001,18.98
1,I0002,21.18
2,I0003,16.19
3,I0004,22.4
4,I0005,17.7
5,I0006,17.96
6,I0007,8.15
7,I0008,20.19
8,I0009,24.61
9,I0010,22.63


This checks the costs against the generator's rules for Books, DVDs and Audiobooks.

In [53]:
items_with_type = silver_items.merge(
    item_rules[["item_id", "item_type"]],
    on="item_id",
    how="left"
)

cost_ranges = {
    "Book": (6, 25),
    "DVD": (8, 20),
    "Audiobook": (12, 30)
}

items_with_type["cost_out_of_range"] = items_with_type.apply(
    lambda row: (
        pd.notna(row["replacement_cost"]) and
        not (
            cost_ranges[row["item_type"]][0]
            <= row["replacement_cost"]
            <= cost_ranges[row["item_type"]][1]
        )
    ),
    axis=1
)

out_of_range = items_with_type[
    items_with_type["cost_out_of_range"]
]

print("Costs outside expected ranges:", len(out_of_range))
display(out_of_range)

Costs outside expected ranges: 0


,item_id,title,classification,replacement_cost,home_branch_id,item_type,cost_out_of_range


Before we save the Silver items table, let's check whether the original classification values match the reference mapping.

In [54]:
# Compare original classifications with the reference mapping
classification_check = silver_items.merge(
    item_rules[["item_id", "classification", "item_type", "category"]],
    on="item_id",
    how="left",
    suffixes=("_raw", "_rule")
)

mismatches = classification_check[
    classification_check["classification_raw"]
    != classification_check["classification_rule"]
]

print("Classification mismatches:", len(mismatches))
display(mismatches.head(20))

print("\nOriginal classification values:")
display(silver_items["classification"].value_counts(dropna=False))

print("\nReference classification values:")
display(item_rules["classification"].value_counts(dropna=False))

Classification mismatches: 0


,item_id,title,classification_raw,replacement_cost,home_branch_id,classification_rule,item_type,category



Original classification values:


,count
classification,
Book / Fiction,306
Book / Non-fiction,247
Book / Children,200
DVD / Film,98
Audiobook / Fiction,64
DVD / Children,61
Audiobook / Non-fiction,45
Book / Reference,35
Book / fiction,18



Reference classification values:


,count
classification,
Book / Fiction,306
Book / Non-fiction,247
Book / Children,200
DVD / Film,98
Audiobook / Fiction,64
DVD / Children,61
Audiobook / Non-fiction,45
Book / Reference,35
Book / fiction,18


Great result! Classification mismatches: 0. Every item’s original classification matches the reference mapping exactly.

However, the value counts show that the classifications have inconsistent formatting, such as:
- Book / Fiction, Book / fiction, and Book / FICTION
- Book / Non-fiction, Book / nonfiction, and Book / Non fiction
- DVD / Film and DVD / Films
- Book / Children and Book / Childrens

I'll use the reference mapping's item_type and category fields to create consistent values, while keeping the original classification column unchanged.

In [55]:
# Add standardised item type and category from the reference mapping
silver_items = silver_items.merge(
    item_rules[["item_id", "item_type", "category"]],
    on="item_id",
    how="left"
)

# Standardise the category labels
category_map = {
    "fiction": "Fiction",
    "non-fiction": "Non-fiction",
    "children": "Children",
    "childrens": "Children",
    "children's": "Children",
    "film": "Film",
    "films": "Film",
    "reference": "Reference",
    "ref.": "Reference"
}

silver_items["category"] = (
    silver_items["category"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(category_map)
)

# Create a consistent classification column
silver_items["standardised_classification"] = (
    silver_items["item_type"] + " / " + silver_items["category"]
)

print("Missing item types:", silver_items["item_type"].isna().sum())
print("Missing categories:", silver_items["category"].isna().sum())

print("\nStandardised classifications:")
display(silver_items["standardised_classification"].value_counts(dropna=False))

Missing item types: 0
Missing categories: 0

Standardised classifications:


,count
standardised_classification,
Book / Fiction,347
Book / Non-fiction,276
Book / Children,234
DVD / Film,111
Audiobook / Fiction,74
DVD / Children,65
Audiobook / Non-fiction,51
Book / Reference,42


Excellent!! Let's check for duplicate IDs, missing values, and whether the row count is still correct.

In [56]:
print("Silver items shape:", silver_items.shape)

print(
    "Duplicate item IDs:",
    silver_items["item_id"].duplicated().sum()
)

print("\nMissing values:")
display(silver_items.isna().sum())

print(
    "\nOriginal item count:",
    len(items)
)

print(
    "Silver item count:",
    len(silver_items)
)

print(
    "\nUnique standardised classifications:",
    silver_items["standardised_classification"].nunique()
)

Silver items shape: (1200, 8)
Duplicate item IDs: 0

Missing values:


,0
item_id,0
title,0
classification,0
replacement_cost,24
home_branch_id,0
item_type,0
category,0
standardised_classification,0



Original item count: 1200
Silver item count: 1200

Unique standardised classifications: 8


Brilliant! Silver items validation has passed! The 24 missing replacement costs are the only remaining missing values. We'll retain them as missing rather than make up values.

Now, i gonna save the silver items CSV.

In [57]:
from pathlib import Path

silver_path = Path("data/silver")
silver_path.mkdir(parents=True, exist_ok=True)

silver_items.to_csv(
    silver_path / "silver_items.csv",
    index=False
)

print("Saved:", silver_path / "silver_items.csv")

Saved: data/silver/silver_items.csv


Then verify that the file exists and  can be read.

In [58]:
check_silver_items = pd.read_csv(
    silver_path / "silver_items.csv"
)

print("Saved file shape:", check_silver_items.shape)
display(check_silver_items.head())

Saved file shape: (1200, 8)


,item_id,title,classification,replacement_cost,home_branch_id,item_type,category,standardised_classification
0,I0001,The Secret Map,Book / Children,18.98,B03,Book,Children,Book / Children
1,I0002,The Quiet Season,Book / Fiction,21.18,B05,Book,Fiction,Book / Fiction
2,I0003,The Hidden River,Book / Non-fiction,16.19,B01,Book,Non-fiction,Book / Non-fiction
3,I0004,The Paper Letters,Book / Non-fiction,22.40,B01,Book,Non-fiction,Book / Non-fiction
4,I0005,The Quiet Orchard,Book / Childrens,17.70,B05,Book,Children,Book / Children
